# 02 Data Cleaning

## Objective
Standardize station names and connection labels so that nodes and edges match correctly in the graph.

## Inputs
- `data/processed/tfl_stations.csv`
- `data/processed/tube_connections.csv`

## Outputs
- `data/processed/stations_clean.csv`
- `data/processed/connections_clean.csv`

## Why this step matters
Graph-based analysis depends on exact matching between station names in the node table and station names in the edge table. Even small naming differences can create isolated nodes and incorrect network fragmentation.

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

stations = pd.read_csv(PROCESSED_DIR / "tfl_stations.csv")
connections = pd.read_csv(PROCESSED_DIR / "tube_connections.csv")

print("stations shape:", stations.shape)
print("connections shape:", connections.shape)
stations.head()

## Step 1: Define a common station-name cleaning function

This function standardizes station names across both datasets by:
- converting text to lowercase
- removing suffixes like `station` and `underground`
- removing apostrophes
- handling symbols like `&`
- trimming extra spaces

In [ ]:
def clean_station_name(name):
    if pd.isna(name):
        return name

    name = str(name).lower().strip()
    name = name.replace(" station", "")
    name = name.replace(" underground", "")
    name = name.replace(" london underground ltd", "")
    name = name.replace("'", "")
    name = name.replace("&", "and")

    if "/" in name:
        name = name.split("/")[0].strip()

    name = " ".join(name.split())
    return name

In [ ]:
stations["station_name_clean"] = stations["station_name"].apply(clean_station_name)
stations[["station_name", "station_name_clean"]].head(20)

## Step 2: Map route connection IDs to station names

The route file contains station IDs rather than final names. This step maps each connection ID to its station name.

In [ ]:
id_to_name = stations.set_index("station_id")["station_name"].to_dict()

connections["from_name"] = connections["from_id"].map(id_to_name)
connections["to_name"] = connections["to_id"].map(id_to_name)

connections[["from_id", "from_name", "to_id", "to_name"]].head()

In [ ]:
connections["from_name_clean"] = connections["from_name"].apply(clean_station_name)
connections["to_name_clean"] = connections["to_name"].apply(clean_station_name)

connections[["from_name", "from_name_clean", "to_name", "to_name_clean"]].head(20)

## Step 3: Check for naming mismatches

This step verifies whether cleaned connection names exist in the cleaned station name set.

In [ ]:
station_name_set = set(stations["station_name_clean"])

from_not_in_stations = set(connections["from_name_clean"]) - station_name_set
to_not_in_stations = set(connections["to_name_clean"]) - station_name_set

print("from mismatches:", len(from_not_in_stations))
print("to mismatches:", len(to_not_in_stations))
print("sample from mismatches:", list(from_not_in_stations)[:20])
print("sample to mismatches:", list(to_not_in_stations)[:20])

In [ ]:
stations_fixed = stations.copy()
stations_fixed["station_name"] = stations_fixed["station_name_clean"]

connections_fixed = connections.copy()
connections_fixed["from_name"] = connections_fixed["from_name_clean"]
connections_fixed["to_name"] = connections_fixed["to_name_clean"]

In [ ]:
stations_fixed = stations_fixed.dropna(subset=["station_id", "station_name", "lat", "lon"])
connections_fixed = connections_fixed.dropna(subset=["from_name", "to_name"])

stations_fixed = stations_fixed.drop_duplicates(subset=["station_name"])
connections_fixed = connections_fixed.drop_duplicates(subset=["from_name", "to_name", "line"])

## Step 4: Save cleaned station and connection datasets

These files are the final network inputs used in graph construction and simulation.

In [ ]:
stations_fixed.to_csv(PROCESSED_DIR / "stations_clean.csv", index=False)
connections_fixed.to_csv(PROCESSED_DIR / "connections_clean.csv", index=False)

print("Saved corrected stations_clean.csv")
print("Saved corrected connections_clean.csv")